# Phase 4 : Transformation et Feature Engineering

## Objectif

Combiner les sources NBA nettoyées et créer des variables dérivées pertinentes
pour prédire les points par match d'un joueur lors de la saison suivante.

La granularité cible du dataset final est :

`1 ligne = 1 joueur NBA pour 1 saison`

La clé principale visée est :

`PLAYER_ID + SEASON`

La Phase 4 suit l'ordre du guide :

1. préparation et validation des jointures ;
2. combinaison des sources ;
3. création des features ;
4. sélection et organisation des colonnes ;
5. documentation et validation du dataset final.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 120)


PROJECT_ROOT = Path.cwd()

if (
    not (PROJECT_ROOT / "data").exists()
    and (PROJECT_ROOT / "ML_NBA_Project" / "data").exists()
):
    PROJECT_ROOT = PROJECT_ROOT / "ML_NBA_Project"


CLEAN_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "phase4_transformation"
)


PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Projet :", PROJECT_ROOT.resolve())
print("Données nettoyées :", CLEAN_DIR.resolve())
print("Données transformées :", PROCESSED_DIR.resolve())
print("Rapports :", REPORT_DIR.resolve())

Projet : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model
Données nettoyées : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean
Données transformées : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\processed
Rapports : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\reports\phase4_transformation


In [2]:
clean_files = {
    "player_bio":
        CLEAN_DIR / "player_bio_clean.parquet",

    "player_stats_base":
        CLEAN_DIR / "player_stats_base_clean.parquet",

    "player_stats_advanced":
        CLEAN_DIR / "player_stats_advanced_clean.parquet",

    "team_stats_base":
        CLEAN_DIR / "team_stats_base_clean.parquet",

    "team_stats_advanced":
        CLEAN_DIR / "team_stats_advanced_clean.parquet",

    "rosters":
        CLEAN_DIR / "rosters_clean.parquet",

    "kaggle_all_seasons":
        CLEAN_DIR / "kaggle_all_seasons_clean.parquet",
}


sources = {}


for source_name, path in clean_files.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Fichier introuvable : {path}"
        )

    sources[source_name] = pd.read_parquet(
        path
    )


source_manifest = pd.DataFrame([
    {
        "source": source_name,
        "lignes": len(df),
        "colonnes": df.shape[1],
    }
    for source_name, df in sources.items()
])


display(source_manifest)

,source,lignes,colonnes
0,player_bio,12846,31
1,player_stats_base,12846,69
2,player_stats_advanced,12846,81
3,team_stats_base,802,56
4,team_stats_advanced,802,48
5,rosters,11672,23
6,kaggle_all_seasons,12844,24


## 1. Jointures entre les sources

Avant toute combinaison, les clés de jointure sont vérifiées.

Pour chaque source, nous contrôlons :

- la présence des colonnes constituant la clé ;
- le type des clés ;
- les valeurs manquantes dans les clés ;
- l'unicité ;
- la cardinalité attendue de la future jointure.

Aucune jointure n'est réalisée avant validation de ces contrôles.

In [3]:
SOURCE_KEYS = {
    "player_bio": [
        "PLAYER_ID",
        "SEASON",
    ],

    "player_stats_base": [
        "PLAYER_ID",
        "SEASON",
    ],

    "player_stats_advanced": [
        "PLAYER_ID",
        "SEASON",
    ],

    "team_stats_base": [
        "TEAM_ID",
        "SEASON",
    ],

    "team_stats_advanced": [
        "TEAM_ID",
        "SEASON",
    ],

    "rosters": [
        "PLAYER_ID",
        "TeamID",
        "SEASON",
    ],

    "kaggle_all_seasons": [
        "player_name",
        "season",
    ],
}

In [4]:
key_audit_rows = []


for source_name, keys in SOURCE_KEYS.items():

    df = sources[source_name]

    missing_columns = [
        col
        for col in keys
        if col not in df.columns
    ]

    if missing_columns:

        key_audit_rows.append({
            "source": source_name,
            "cle": " + ".join(keys),
            "lignes": len(df),
            "colonnes_cle_presentes": False,
            "types": "Colonnes manquantes",
            "valeurs_manquantes_cle": np.nan,
            "lignes_cle_dupliquees": np.nan,
            "groupes_dupliques": np.nan,
            "cles_uniques": np.nan,
        })

        continue


    null_key_rows = int(
        df[keys]
        .isna()
        .any(axis=1)
        .sum()
    )


    duplicate_mask = df.duplicated(
        subset=keys,
        keep=False
    )


    duplicate_groups = (
        df.loc[
            duplicate_mask,
            keys
        ]
        .drop_duplicates()
        .shape[0]
    )


    key_types = " | ".join(
        f"{col}:{df[col].dtype}"
        for col in keys
    )


    key_audit_rows.append({
        "source": source_name,
        "cle": " + ".join(keys),
        "lignes": len(df),
        "colonnes_cle_presentes": True,
        "types": key_types,
        "valeurs_manquantes_cle": null_key_rows,
        "lignes_cle_dupliquees": int(
            duplicate_mask.sum()
        ),
        "groupes_dupliques": int(
            duplicate_groups
        ),
        "cles_uniques": int(
            df[keys]
            .drop_duplicates()
            .shape[0]
        ),
    })


key_audit = pd.DataFrame(
    key_audit_rows
)


key_audit.to_csv(
    REPORT_DIR
    / "01_audit_cles_jointure.csv",
    index=False,
    encoding="utf-8"
)


display(key_audit)

,source,cle,lignes,colonnes_cle_presentes,types,valeurs_manquantes_cle,lignes_cle_dupliquees,groupes_dupliques,cles_uniques
0,player_bio,PLAYER_ID + SEASON,12846,True,PLAYER_ID:int64 | SEASON:string,0,0,0,12846
1,player_stats_base,PLAYER_ID + SEASON,12846,True,PLAYER_ID:int64 | SEASON:string,0,0,0,12846
2,player_stats_advanced,PLAYER_ID + SEASON,12846,True,PLAYER_ID:int64 | SEASON:string,0,0,0,12846
3,team_stats_base,TEAM_ID + SEASON,802,True,TEAM_ID:int64 | SEASON:string,0,0,0,802
4,team_stats_advanced,TEAM_ID + SEASON,802,True,TEAM_ID:int64 | SEASON:string,0,0,0,802
5,rosters,PLAYER_ID + TeamID + SEASON,11672,True,PLAYER_ID:int64 | TeamID:int64 | SEASON:string,0,0,0,11672
6,kaggle_all_seasons,player_name + season,12844,True,player_name:object | season:object,0,8,4,12840


In [5]:
player_sources = [
    "player_bio",
    "player_stats_base",
    "player_stats_advanced",
]


player_key_sets = {}


for source_name in player_sources:

    df = sources[source_name]

    player_key_sets[
        source_name
    ] = set(
        map(
            tuple,
            df[
                [
                    "PLAYER_ID",
                    "SEASON",
                ]
            ].to_numpy()
        )
    )


player_alignment_rows = []


reference_source = "player_stats_base"
reference_keys = player_key_sets[
    reference_source
]


for source_name in player_sources:

    current_keys = player_key_sets[
        source_name
    ]

    player_alignment_rows.append({
        "reference":
            reference_source,

        "source_comparee":
            source_name,

        "cles_reference":
            len(reference_keys),

        "cles_source":
            len(current_keys),

        "absentes_de_source":
            len(
                reference_keys
                - current_keys
            ),

        "absentes_reference":
            len(
                current_keys
                - reference_keys
            ),

        "alignement_parfait":
            reference_keys
            == current_keys,
    })


player_alignment = pd.DataFrame(
    player_alignment_rows
)


display(player_alignment)


player_alignment.to_csv(
    REPORT_DIR
    / "01_alignement_sources_joueurs.csv",
    index=False,
    encoding="utf-8"
)

,reference,source_comparee,cles_reference,cles_source,absentes_de_source,absentes_reference,alignement_parfait
0,player_stats_base,player_bio,12846,12846,0,0,True
1,player_stats_base,player_stats_base,12846,12846,0,0,True
2,player_stats_base,player_stats_advanced,12846,12846,0,0,True


In [6]:
team_key_sets = {}


for source_name in [
    "team_stats_base",
    "team_stats_advanced",
]:

    df = sources[source_name]

    team_key_sets[
        source_name
    ] = set(
        map(
            tuple,
            df[
                [
                    "TEAM_ID",
                    "SEASON",
                ]
            ].to_numpy()
        )
    )


base_team_keys = team_key_sets[
    "team_stats_base"
]

advanced_team_keys = team_key_sets[
    "team_stats_advanced"
]


team_alignment = pd.DataFrame([
    {
        "source_1":
            "team_stats_base",

        "source_2":
            "team_stats_advanced",

        "cles_source_1":
            len(base_team_keys),

        "cles_source_2":
            len(advanced_team_keys),

        "uniquement_source_1":
            len(
                base_team_keys
                - advanced_team_keys
            ),

        "uniquement_source_2":
            len(
                advanced_team_keys
                - base_team_keys
            ),

        "alignement_parfait":
            base_team_keys
            == advanced_team_keys,
    }
])


display(team_alignment)

,source_1,source_2,cles_source_1,cles_source_2,uniquement_source_1,uniquement_source_2,alignement_parfait
0,team_stats_base,team_stats_advanced,802,802,0,0,True


In [7]:
candidate_team_columns = []


for source_name in [
    "player_bio",
    "player_stats_base",
    "player_stats_advanced",
]:

    df = sources[source_name]

    matches = [
        col
        for col in df.columns
        if (
            "TEAM" in col.upper()
            or col.upper() == "TEAMID"
        )
    ]

    for col in matches:

        candidate_team_columns.append({
            "source": source_name,
            "colonne": col,
            "dtype": str(
                df[col].dtype
            ),
            "valeurs_uniques": int(
                df[col].nunique(
                    dropna=True
                )
            ),
            "manquants": int(
                df[col].isna().sum()
            ),
        })


candidate_team_columns = pd.DataFrame(
    candidate_team_columns
)


display(candidate_team_columns)

,source,colonne,dtype,valeurs_uniques,manquants
0,player_bio,TEAM_ID,int64,30,0
1,player_bio,TEAM_ABBREVIATION,object,36,0
2,player_stats_base,TEAM_ID,int64,30,0
3,player_stats_base,TEAM_ABBREVIATION,object,36,0
4,player_stats_base,TEAM_COUNT,int64,5,0
5,player_stats_advanced,TEAM_ID,int64,30,0
6,player_stats_advanced,TEAM_ABBREVIATION,object,36,0
7,player_stats_advanced,TEAM_COUNT,int64,5,0


In [8]:
rosters = sources[
    "rosters"
]


roster_player_season = (
    rosters
    .groupby(
        [
            "PLAYER_ID",
            "SEASON",
        ]
    )
    .agg(
        nombre_lignes=(
            "TeamID",
            "size"
        ),
        nombre_equipes=(
            "TeamID",
            "nunique"
        ),
    )
    .reset_index()
)


roster_multi = roster_player_season[
    roster_player_season[
        "nombre_lignes"
    ] > 1
]


print(
    "Joueur-saisons avec plusieurs lignes roster :",
    len(roster_multi)
)


display(
    roster_multi.head(20)
)

Joueur-saisons avec plusieurs lignes roster : 1


,PLAYER_ID,SEASON,nombre_lignes,nombre_equipes
9348,203915,2015-16,2,2


### Cas particulier du dataset Kaggle

Le dataset Kaggle ne contient pas `PLAYER_ID`.

La clé candidate `player_name + season` n'est pas unique : le diagnostic a
identifié plusieurs homonymes partageant le même nom au cours d'une même
saison.

Une jointure directe uniquement sur le nom et la saison risquerait donc
d'associer des statistiques au mauvais joueur.

Le dataset Kaggle ne sera pas joint aux sources NBA API tant qu'un mapping
fiable vers `PLAYER_ID` n'aura pas été construit ou tant que son apport au
dataset final n'aura pas été jugé suffisamment pertinent pour justifier ce
mapping.

### Validation des relations entre les sources

Les clés principales sont uniques et correctement alignées.

Avant d'effectuer les jointures, trois contrôles complémentaires sont réalisés :

- vérifier que les trois sources joueur attribuent le même `TEAM_ID` à chaque
  `PLAYER_ID + SEASON` ;
- vérifier que les équipes associées aux joueurs existent dans les tables
  statistiques équipe ;
- identifier les colonnes présentes dans plusieurs sources afin d'éviter les
  duplications inutiles lors des jointures.

In [9]:
# Cohérence des équipes entre les sources joueur

team_consistency = (
    sources["player_stats_base"][
        ["PLAYER_ID", "SEASON", "TEAM_ID"]
    ]
    .rename(
        columns={
            "TEAM_ID": "TEAM_ID_BASE"
        }
    )
    .merge(
        sources["player_bio"][
            ["PLAYER_ID", "SEASON", "TEAM_ID"]
        ].rename(
            columns={
                "TEAM_ID": "TEAM_ID_BIO"
            }
        ),
        on=[
            "PLAYER_ID",
            "SEASON",
        ],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        sources["player_stats_advanced"][
            ["PLAYER_ID", "SEASON", "TEAM_ID"]
        ].rename(
            columns={
                "TEAM_ID": "TEAM_ID_ADVANCED"
            }
        ),
        on=[
            "PLAYER_ID",
            "SEASON",
        ],
        how="inner",
        validate="one_to_one",
    )
)


team_consistency[
    "base_bio_identiques"
] = (
    team_consistency[
        "TEAM_ID_BASE"
    ]
    == team_consistency[
        "TEAM_ID_BIO"
    ]
)


team_consistency[
    "base_advanced_identiques"
] = (
    team_consistency[
        "TEAM_ID_BASE"
    ]
    == team_consistency[
        "TEAM_ID_ADVANCED"
    ]
)


team_consistency_summary = pd.DataFrame([
    {
        "controle":
            "TEAM_ID base vs bio",
        "differences":
            int(
                (
                    ~team_consistency[
                        "base_bio_identiques"
                    ]
                ).sum()
            ),
    },
    {
        "controle":
            "TEAM_ID base vs advanced",
        "differences":
            int(
                (
                    ~team_consistency[
                        "base_advanced_identiques"
                    ]
                ).sum()
            ),
    },
])


display(
    team_consistency_summary
)

,controle,differences
0,TEAM_ID base vs bio,0
1,TEAM_ID base vs advanced,0


In [10]:
# Vérification de la correspondance joueur-équipe

player_team_keys = (
    sources[
        "player_stats_base"
    ][
        [
            "TEAM_ID",
            "SEASON",
        ]
    ]
    .drop_duplicates()
)


team_base_keys = (
    sources[
        "team_stats_base"
    ][
        [
            "TEAM_ID",
            "SEASON",
        ]
    ]
    .drop_duplicates()
)


team_advanced_keys = (
    sources[
        "team_stats_advanced"
    ][
        [
            "TEAM_ID",
            "SEASON",
        ]
    ]
    .drop_duplicates()
)


coverage_base = player_team_keys.merge(
    team_base_keys,
    on=[
        "TEAM_ID",
        "SEASON",
    ],
    how="left",
    indicator=True,
)


coverage_advanced = player_team_keys.merge(
    team_advanced_keys,
    on=[
        "TEAM_ID",
        "SEASON",
    ],
    how="left",
    indicator=True,
)


team_coverage = pd.DataFrame([
    {
        "source_equipe":
            "team_stats_base",
        "cles_joueur_equipe":
            len(player_team_keys),
        "cles_non_trouvees":
            int(
                (
                    coverage_base[
                        "_merge"
                    ]
                    != "both"
                ).sum()
            ),
    },
    {
        "source_equipe":
            "team_stats_advanced",
        "cles_joueur_equipe":
            len(player_team_keys),
        "cles_non_trouvees":
            int(
                (
                    coverage_advanced[
                        "_merge"
                    ]
                    != "both"
                ).sum()
            ),
    },
])


display(team_coverage)

,source_equipe,cles_joueur_equipe,cles_non_trouvees
0,team_stats_base,802,0
1,team_stats_advanced,802,0


In [11]:
# Joueurs ayant plusieurs équipes dans une saison

team_count_distribution = (
    sources[
        "player_stats_base"
    ]["TEAM_COUNT"]
    .value_counts()
    .sort_index()
    .rename_axis(
        "TEAM_COUNT"
    )
    .reset_index(
        name="joueur_saisons"
    )
)


display(
    team_count_distribution
)


multi_team_players = (
    sources[
        "player_stats_base"
    ]
    .loc[
        sources[
            "player_stats_base"
        ]["TEAM_COUNT"] > 1,
        [
            "PLAYER_ID",
            "PLAYER_NAME",
            "SEASON",
            "TEAM_ID",
            "TEAM_ABBREVIATION",
            "TEAM_COUNT",
        ]
    ]
    .sort_values(
        [
            "SEASON",
            "PLAYER_NAME",
        ]
    )
)


print(
    "Joueur-saisons avec TEAM_COUNT > 1 :",
    len(multi_team_players)
)


display(
    multi_team_players.head(20)
)

,TEAM_COUNT,joueur_saisons
0,1,11295
1,2,1452
2,3,96
3,4,2
4,5,1


Joueur-saisons avec TEAM_COUNT > 1 : 1551


,PLAYER_ID,PLAYER_NAME,SEASON,TEAM_ID,TEAM_ABBREVIATION,TEAM_COUNT
0,920,A.C. Green,1996-97,1610612742,DAL,2
1,243,Aaron McKie,1996-97,1610612765,DET,2
2,1425,Aaron Williams,1996-97,1610612763,VAN,2
3,768,Acie Earl,1996-97,1610612749,MIL,2
5,154,Adrian Caldwell,1996-97,1610612755,PHI,2
16,924,Anthony Goldwire,1996-97,1610612743,DEN,2
51,240,Brooks Thompson,1996-97,1610612743,DEN,2
62,76,Cedric Ceballos,1996-97,1610612756,PHX,2
72,423,Chris Gatling,1996-97,1610612751,NJN,2
81,359,Chucky Brown,1996-97,1610612749,MIL,2


In [12]:
# Correspondance entre les joueurs et le roster

roster_for_join = (
    sources[
        "rosters"
    ]
    .rename(
        columns={
            "TeamID": "TEAM_ID"
        }
    )
)


roster_match = (
    sources[
        "player_stats_base"
    ][
        [
            "PLAYER_ID",
            "TEAM_ID",
            "SEASON",
        ]
    ]
    .merge(
        roster_for_join[
            [
                "PLAYER_ID",
                "TEAM_ID",
                "SEASON",
            ]
        ],
        on=[
            "PLAYER_ID",
            "TEAM_ID",
            "SEASON",
        ],
        how="left",
        indicator=True,
        validate="one_to_one",
    )
)


roster_coverage = (
    roster_match[
        "_merge"
    ]
    .value_counts()
    .rename_axis(
        "statut"
    )
    .reset_index(
        name="lignes"
    )
)


display(
    roster_coverage
)


print(
    "Taux de correspondance roster :",
    round(
        (
            roster_match[
                "_merge"
            ]
            == "both"
        ).mean()
        * 100,
        2
    ),
    "%"
)

,statut,lignes
0,both,11369
1,left_only,1477
2,right_only,0


Taux de correspondance roster : 88.5 %


In [13]:
# Colonnes communes entre les sources joueur

player_overlap_rows = []


player_source_pairs = [
    (
        "player_stats_base",
        "player_bio",
    ),
    (
        "player_stats_base",
        "player_stats_advanced",
    ),
    (
        "player_bio",
        "player_stats_advanced",
    ),
]


for source_1, source_2 in player_source_pairs:

    common_columns = sorted(
        set(
            sources[
                source_1
            ].columns
        )
        &
        set(
            sources[
                source_2
            ].columns
        )
    )

    for col in common_columns:

        if col in {
            "PLAYER_ID",
            "SEASON",
        }:
            continue

        player_overlap_rows.append({
            "source_1": source_1,
            "source_2": source_2,
            "colonne": col,
        })


player_column_overlap = pd.DataFrame(
    player_overlap_rows
)


print(
    "Chevauchements de colonnes entre sources joueur :",
    len(player_column_overlap)
)


display(
    player_column_overlap
)

Chevauchements de colonnes entre sources joueur : 44


,source_1,source_2,colonne
0,player_stats_base,player_bio,AGE
1,player_stats_base,player_bio,AST
2,player_stats_base,player_bio,GP
3,player_stats_base,player_bio,PLAYER_NAME
4,player_stats_base,player_bio,PTS
5,player_stats_base,player_bio,REB
6,player_stats_base,player_bio,SOURCE_FILE
7,player_stats_base,player_bio,TEAM_ABBREVIATION
8,player_stats_base,player_bio,TEAM_ID
9,player_stats_base,player_stats_advanced,AGE


In [14]:
# Colonnes communes entre les sources équipe

team_common_columns = sorted(
    set(
        sources[
            "team_stats_base"
        ].columns
    )
    &
    set(
        sources[
            "team_stats_advanced"
        ].columns
    )
)


team_column_overlap = pd.DataFrame({
    "colonne":
        team_common_columns
})


display(
    team_column_overlap
)

,colonne
0,GP
1,GP_RANK
2,L
3,L_RANK
4,MIN
5,MIN_RANK
6,SEASON
7,SOURCE_FILE
8,TEAM_ID
9,TEAM_NAME


In [15]:
# Diagnostic des non-correspondances roster

player_team_keys = (
    sources["player_stats_base"][
        [
            "PLAYER_ID",
            "TEAM_ID",
            "SEASON",
        ]
    ]
    .copy()
)


roster_keys = (
    sources["rosters"]
    .rename(
        columns={
            "TeamID": "TEAM_ID"
        }
    )[
        [
            "PLAYER_ID",
            "TEAM_ID",
            "SEASON",
        ]
    ]
    .copy()
)


exact_match = player_team_keys.merge(
    roster_keys,
    on=[
        "PLAYER_ID",
        "TEAM_ID",
        "SEASON",
    ],
    how="left",
    indicator=True,
)


exact_missing = exact_match[
    exact_match["_merge"] == "left_only"
][
    [
        "PLAYER_ID",
        "TEAM_ID",
        "SEASON",
    ]
]


roster_player_season = (
    roster_keys[
        [
            "PLAYER_ID",
            "SEASON",
        ]
    ]
    .drop_duplicates()
)


fallback_check = exact_missing.merge(
    roster_player_season,
    on=[
        "PLAYER_ID",
        "SEASON",
    ],
    how="left",
    indicator=True,
)


roster_diagnostic = pd.DataFrame([
    {
        "categorie":
            "Correspondance exacte joueur + équipe + saison",
        "lignes":
            int(
                (
                    exact_match["_merge"]
                    == "both"
                ).sum()
            ),
    },
    {
        "categorie":
            "Même joueur-saison mais équipe différente",
        "lignes":
            int(
                (
                    fallback_check["_merge"]
                    == "both"
                ).sum()
            ),
    },
    {
        "categorie":
            "Joueur-saison totalement absent du roster",
        "lignes":
            int(
                (
                    fallback_check["_merge"]
                    == "left_only"
                ).sum()
            ),
    },
])


roster_diagnostic["pourcentage"] = (
    roster_diagnostic["lignes"]
    / len(player_team_keys)
    * 100
).round(2)


display(
    roster_diagnostic
)

,categorie,lignes,pourcentage
0,Correspondance exacte joueur + équipe + saison,11369,88.50
1,Même joueur-saison mais équipe différente,75,0.58
2,Joueur-saison totalement absent du roster,1402,10.91


In [16]:
# Non-correspondances roster par saison

roster_missing_by_season = (
    exact_missing
    .groupby(
        "SEASON"
    )
    .size()
    .reset_index(
        name="non_correspondances"
    )
)


season_totals = (
    player_team_keys
    .groupby(
        "SEASON"
    )
    .size()
    .reset_index(
        name="joueur_saisons"
    )
)


roster_missing_by_season = (
    season_totals
    .merge(
        roster_missing_by_season,
        on="SEASON",
        how="left",
    )
)


roster_missing_by_season[
    "non_correspondances"
] = (
    roster_missing_by_season[
        "non_correspondances"
    ]
    .fillna(0)
    .astype(int)
)


roster_missing_by_season[
    "pourcentage"
] = (
    roster_missing_by_season[
        "non_correspondances"
    ]
    / roster_missing_by_season[
        "joueur_saisons"
    ]
    * 100
).round(2)


display(
    roster_missing_by_season
)

,SEASON,joueur_saisons,non_correspondances,pourcentage
0,1996-97,441,90,20.41
1,1997-98,439,91,20.73
2,1998-99,440,94,21.36
3,1999-00,439,96,21.87
4,2000-01,441,90,20.41
5,2001-02,440,74,16.82
6,2002-03,428,63,14.72
7,2003-04,442,37,8.37
8,2004-05,464,86,18.53
9,2005-06,458,51,11.14


### Stratégie de jointure

La granularité cible est une ligne par `PLAYER_ID + SEASON`.

`player_stats_base` est utilisée comme table principale car elle contient
l'ensemble des statistiques classiques nécessaires à la modélisation.

Les enrichissements sont réalisés avec des jointures `left` afin de conserver
les 12 846 joueur-saisons de la table principale.

Les sources sont combinées selon les relations suivantes :

- `player_stats_base` + `player_bio` : jointure 1-1 sur `PLAYER_ID + SEASON`
- ajout de `player_stats_advanced` : jointure 1-1 sur `PLAYER_ID + SEASON`
- ajout des statistiques équipe : jointure N-1 sur `TEAM_ID + SEASON`
- ajout du roster : jointure 1-1 sur `PLAYER_ID + SEASON` après création
  d'un profil joueur-saison

Les colonnes déjà présentes dans une source prioritaire ne sont pas dupliquées.

Le dataset Kaggle n'est pas joint directement car `player_name + season`
n'est pas une clé suffisamment fiable pour identifier un joueur de manière
unique.

In [17]:
# Suivi des jointures

join_log = []


def add_join_log(
    numero,
    table_gauche,
    table_droite,
    cle,
    type_jointure,
    cardinalite,
    lignes_avant,
    lignes_apres,
    correspondances,
    total,
):
    join_log.append({
        "jointure": numero,
        "table_gauche": table_gauche,
        "table_droite": table_droite,
        "cle": cle,
        "type": type_jointure,
        "cardinalite": cardinalite,
        "lignes_avant": lignes_avant,
        "lignes_apres": lignes_apres,
        "correspondances": correspondances,
        "taux_correspondance": round(
            correspondances / total * 100,
            2
        ),
    })

In [18]:
# Table principale joueur-saison

player_dataset = (
    sources[
        "player_stats_base"
    ]
    .copy()
)


print(
    "Dimensions initiales :",
    player_dataset.shape
)


print(
    "Clé unique PLAYER_ID + SEASON :",
    not player_dataset.duplicated(
        subset=[
            "PLAYER_ID",
            "SEASON",
        ]
    ).any()
)

Dimensions initiales : (12846, 69)
Clé unique PLAYER_ID + SEASON : True


In [19]:
# Colonnes biographiques non redondantes

base_columns = set(
    sources[
        "player_stats_base"
    ].columns
)

advanced_columns = set(
    sources[
        "player_stats_advanced"
    ].columns
)


bio_columns_to_add = [
    col
    for col in sources[
        "player_bio"
    ].columns
    if (
        col
        not in base_columns
        and col
        not in advanced_columns
    )
]


bio_columns_to_add = [
    col
    for col in bio_columns_to_add
    if col != "SOURCE_FILE"
]


print(
    "Colonnes biographiques ajoutées :",
    len(bio_columns_to_add)
)


print(
    bio_columns_to_add
)

Colonnes biographiques ajoutées : 14
['PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'PLAYER_HEIGHT_CM', 'PLAYER_WEIGHT_KG', 'COLLEGE_IS_MISSING', 'DRAFT_INFO_IS_MISSING', 'PLAYER_HEIGHT_IS_MISSING', 'PLAYER_WEIGHT_IS_MISSING']


In [20]:
# Jointure avec player_bio

rows_before = len(
    player_dataset
)


bio_for_join = (
    sources[
        "player_bio"
    ][
        [
            "PLAYER_ID",
            "SEASON",
        ]
        + bio_columns_to_add
    ]
)


player_dataset = player_dataset.merge(
    bio_for_join,
    on=[
        "PLAYER_ID",
        "SEASON",
    ],
    how="left",
    validate="one_to_one",
    indicator="_merge_bio",
)


bio_matches = int(
    (
        player_dataset[
            "_merge_bio"
        ]
        == "both"
    ).sum()
)


add_join_log(
    numero=1,
    table_gauche="player_stats_base",
    table_droite="player_bio",
    cle="PLAYER_ID + SEASON",
    type_jointure="left",
    cardinalite="1-1",
    lignes_avant=rows_before,
    lignes_apres=len(player_dataset),
    correspondances=bio_matches,
    total=rows_before,
)


player_dataset = player_dataset.drop(
    columns="_merge_bio"
)


print(
    "Dimensions après player_bio :",
    player_dataset.shape
)

print(
    "Correspondances :",
    bio_matches,
    "/",
    rows_before
)

Dimensions après player_bio : (12846, 83)
Correspondances : 12846 / 12846


In [21]:
# Colonnes avancées non redondantes

advanced_columns_to_add = [
    col
    for col in sources[
        "player_stats_advanced"
    ].columns
    if col not in player_dataset.columns
]


advanced_columns_to_add = [
    col
    for col in advanced_columns_to_add
    if col != "SOURCE_FILE"
]


print(
    "Colonnes avancées ajoutées :",
    len(advanced_columns_to_add)
)

Colonnes avancées ajoutées : 56


In [22]:
# Jointure avec player_stats_advanced

rows_before = len(
    player_dataset
)


advanced_for_join = (
    sources[
        "player_stats_advanced"
    ][
        [
            "PLAYER_ID",
            "SEASON",
        ]
        + advanced_columns_to_add
    ]
)


player_dataset = player_dataset.merge(
    advanced_for_join,
    on=[
        "PLAYER_ID",
        "SEASON",
    ],
    how="left",
    validate="one_to_one",
    indicator="_merge_advanced",
)


advanced_matches = int(
    (
        player_dataset[
            "_merge_advanced"
        ]
        == "both"
    ).sum()
)


add_join_log(
    numero=2,
    table_gauche="dataset joueur",
    table_droite="player_stats_advanced",
    cle="PLAYER_ID + SEASON",
    type_jointure="left",
    cardinalite="1-1",
    lignes_avant=rows_before,
    lignes_apres=len(player_dataset),
    correspondances=advanced_matches,
    total=rows_before,
)


player_dataset = player_dataset.drop(
    columns="_merge_advanced"
)


print(
    "Dimensions après statistiques avancées :",
    player_dataset.shape
)

print(
    "Correspondances :",
    advanced_matches,
    "/",
    rows_before
)

Dimensions après statistiques avancées : (12846, 139)
Correspondances : 12846 / 12846


In [23]:
# Construction de la table équipe

team_dataset = sources[
    "team_stats_base"
].copy()

team_advanced_columns = [
    col
    for col in sources[
        "team_stats_advanced"
    ].columns
    if col not in team_dataset.columns
]

team_advanced_columns = [
    col
    for col in team_advanced_columns
    if col != "SOURCE_FILE"
]

rows_before_team = len(
    team_dataset
)

team_dataset = team_dataset.merge(
    sources[
        "team_stats_advanced"
    ][
        [
            "TEAM_ID",
            "SEASON",
        ]
        + team_advanced_columns
    ],
    on=[
        "TEAM_ID",
        "SEASON",
    ],
    how="left",
    validate="one_to_one",
    indicator="_merge_team_advanced",
)

team_advanced_matches = int(
    (
        team_dataset[
            "_merge_team_advanced"
        ]
        == "both"
    ).sum()
)

add_join_log(
    numero=3,
    table_gauche="team_stats_base",
    table_droite="team_stats_advanced",
    cle="TEAM_ID + SEASON",
    type_jointure="left",
    cardinalite="1-1",
    lignes_avant=rows_before_team,
    lignes_apres=len(team_dataset),
    correspondances=team_advanced_matches,
    total=rows_before_team,
)

team_dataset = team_dataset.drop(
    columns="_merge_team_advanced"
)

assert len(team_dataset) == rows_before_team
assert not team_dataset.duplicated(
    subset=[
        "TEAM_ID",
        "SEASON",
    ]
).any()

print(
    "Dimensions table équipe :",
    team_dataset.shape
)

print(
    "Correspondances team_stats_advanced :",
    team_advanced_matches,
    "/",
    rows_before_team
)


Dimensions table équipe : (802, 90)
Correspondances team_stats_advanced : 802 / 802


In [24]:
# Préfixe des variables équipe

team_key_columns = {
    "TEAM_ID",
    "SEASON",
}


team_rename = {
    col: f"TEAM_{col}"
    for col in team_dataset.columns
    if col not in team_key_columns
}


team_dataset = team_dataset.rename(
    columns=team_rename
)


print(
    "Nombre de variables équipe :",
    len(team_rename)
)

Nombre de variables équipe : 88


In [25]:
# Jointure avec les statistiques équipe

rows_before = len(
    player_dataset
)


player_dataset = player_dataset.merge(
    team_dataset,
    on=[
        "TEAM_ID",
        "SEASON",
    ],
    how="left",
    validate="many_to_one",
    indicator="_merge_team",
)


team_matches = int(
    (
        player_dataset[
            "_merge_team"
        ]
        == "both"
    ).sum()
)


add_join_log(
    numero=4,
    table_gauche="dataset joueur",
    table_droite="team_stats_base + team_stats_advanced",
    cle="TEAM_ID + SEASON",
    type_jointure="left",
    cardinalite="N-1",
    lignes_avant=rows_before,
    lignes_apres=len(player_dataset),
    correspondances=team_matches,
    total=rows_before,
)


player_dataset = player_dataset.drop(
    columns="_merge_team"
)


print(
    "Dimensions après statistiques équipe :",
    player_dataset.shape
)

print(
    "Correspondances :",
    team_matches,
    "/",
    rows_before
)

Dimensions après statistiques équipe : (12846, 227)
Correspondances : 12846 / 12846


In [26]:
# Vérification de la stabilité des informations roster

roster_selected = (
    sources[
        "rosters"
    ][
        [
            "PLAYER_ID",
            "SEASON",
            "POSITION",
            "EXP_YEARS",
        ]
    ]
    .copy()
)


roster_conflicts = (
    roster_selected
    .groupby(
        [
            "PLAYER_ID",
            "SEASON",
        ]
    )
    .agg({
        "POSITION": "nunique",
        "EXP_YEARS": "nunique",
    })
    .reset_index()
)


roster_conflicts = roster_conflicts[
    (
        roster_conflicts[
            "POSITION"
        ] > 1
    )
    |
    (
        roster_conflicts[
            "EXP_YEARS"
        ] > 1
    )
]


print(
    "Joueur-saisons avec informations roster contradictoires :",
    len(roster_conflicts)
)


display(
    roster_conflicts.head(20)
)

Joueur-saisons avec informations roster contradictoires : 0


,PLAYER_ID,SEASON,POSITION,EXP_YEARS


In [27]:
# Profil roster au niveau joueur-saison

roster_profile = (
    roster_selected
    .drop_duplicates(
        subset=[
            "PLAYER_ID",
            "SEASON",
            "POSITION",
            "EXP_YEARS",
        ]
    )
)


assert not roster_profile.duplicated(
    subset=[
        "PLAYER_ID",
        "SEASON",
    ]
).any()


print(
    "Profils roster joueur-saison :",
    len(roster_profile)
)

Profils roster joueur-saison : 11671


In [28]:
# Jointure avec le profil roster

rows_before = len(
    player_dataset
)


player_dataset = player_dataset.merge(
    roster_profile,
    on=[
        "PLAYER_ID",
        "SEASON",
    ],
    how="left",
    validate="one_to_one",
    indicator="_merge_roster",
)


roster_matches = int(
    (
        player_dataset[
            "_merge_roster"
        ]
        == "both"
    ).sum()
)


player_dataset[
    "ROSTER_AVAILABLE"
] = (
    player_dataset[
        "_merge_roster"
    ]
    == "both"
).astype("Int8")


add_join_log(
    numero=5,
    table_gauche="dataset joueur",
    table_droite="rosters",
    cle="PLAYER_ID + SEASON",
    type_jointure="left",
    cardinalite="1-1",
    lignes_avant=rows_before,
    lignes_apres=len(player_dataset),
    correspondances=roster_matches,
    total=rows_before,
)


player_dataset = player_dataset.drop(
    columns="_merge_roster"
)


print(
    "Dimensions après roster :",
    player_dataset.shape
)

print(
    "Correspondances roster :",
    roster_matches,
    "/",
    rows_before
)

print(
    "Taux :",
    round(
        roster_matches
        / rows_before
        * 100,
        2
    ),
    "%"
)

Dimensions après roster : (12846, 230)
Correspondances roster : 11444 / 12846
Taux : 89.09 %


In [29]:
# Documentation des jointures

join_report = (
    pd.DataFrame(
        join_log
    )
    .sort_values(
        "jointure"
    )
    .reset_index(
        drop=True
    )
)

join_report.to_csv(
    REPORT_DIR
    / "02_documentation_jointures.csv",
    index=False,
    encoding="utf-8"
)

display(
    join_report
)


,jointure,table_gauche,table_droite,cle,type,cardinalite,lignes_avant,lignes_apres,correspondances,taux_correspondance
0,1,player_stats_base,player_bio,PLAYER_ID + SEASON,left,1-1,12846,12846,12846,100.00
1,2,dataset joueur,player_stats_advanced,PLAYER_ID + SEASON,left,1-1,12846,12846,12846,100.00
2,3,team_stats_base,team_stats_advanced,TEAM_ID + SEASON,left,1-1,802,802,802,100.00
3,4,dataset joueur,team_stats_base + team_stats_advanced,TEAM_ID + SEASON,left,N-1,12846,12846,12846,100.00
4,5,dataset joueur,rosters,PLAYER_ID + SEASON,left,1-1,12846,12846,11444,89.09


In [30]:
# Validation des jointures

join_validation = pd.DataFrame([
    {
        "controle": "Nombre de jointures documentées",
        "valeur": len(join_report),
        "attendu": 5,
        "valide": len(join_report) == 5,
    },
    {
        "controle": "Aucune variation inattendue du nombre de lignes",
        "valeur": bool(
            (
                join_report["lignes_avant"]
                == join_report["lignes_apres"]
            ).all()
        ),
        "attendu": True,
        "valide": bool(
            (
                join_report["lignes_avant"]
                == join_report["lignes_apres"]
            ).all()
        ),
    },
    {
        "controle": "Nombre de lignes final",
        "valeur": len(player_dataset),
        "attendu": 12846,
        "valide": len(player_dataset) == 12846,
    },
    {
        "controle": "Doublons PLAYER_ID + SEASON",
        "valeur": int(
            player_dataset.duplicated(
                subset=[
                    "PLAYER_ID",
                    "SEASON",
                ]
            ).sum()
        ),
        "attendu": 0,
        "valide": not player_dataset.duplicated(
            subset=[
                "PLAYER_ID",
                "SEASON",
            ]
        ).any(),
    },
    {
        "controle": "Correspondances player_bio",
        "valeur": bio_matches,
        "attendu": 12846,
        "valide": bio_matches == 12846,
    },
    {
        "controle": "Correspondances player_stats_advanced",
        "valeur": advanced_matches,
        "attendu": 12846,
        "valide": advanced_matches == 12846,
    },
    {
        "controle": "Correspondances statistiques équipe",
        "valeur": team_matches,
        "attendu": 12846,
        "valide": team_matches == 12846,
    },
])

display(
    join_validation
)

assert join_validation[
    "valide"
].all()

print(
    "Toutes les jointures sont valides."
)


,controle,valeur,attendu,valide
0,Nombre de jointures documentées,5,5,True
1,Aucune variation inattendue du nombre de lignes,True,True,True
2,Nombre de lignes final,12846,12846,True
3,Doublons PLAYER_ID + SEASON,0,0,True
4,Correspondances player_bio,12846,12846,True
5,Correspondances player_stats_advanced,12846,12846,True
6,Correspondances statistiques équipe,12846,12846,True


Toutes les jointures sont valides.


### Résultat des jointures

Les sources statistiques joueur sont parfaitement alignées sur
`PLAYER_ID + SEASON` et ont été combinées sans perte ni multiplication de
lignes.

Les statistiques équipe sont disponibles pour l'ensemble des joueur-saisons et
ont été ajoutées sur `TEAM_ID + SEASON`.

Les colonnes équipe ont été préfixées afin de distinguer clairement les
statistiques individuelles des statistiques collectives.

Le roster présente une couverture plus faible. Les attributs indépendants de
l'équipe ont donc été ramenés au grain `PLAYER_ID + SEASON` avant la jointure.
Les informations absentes sont conservées comme valeurs manquantes et un
indicateur `ROSTER_AVAILABLE` permet de documenter leur disponibilité.

Le dataset Kaggle n'a pas été intégré à ce stade car sa clé
`player_name + season` présente des homonymes et ne permet pas une association
suffisamment sûre avec `PLAYER_ID`.

La granularité du dataset reste :

`1 ligne = 1 PLAYER_ID + SEASON`

## 2. Feature Engineering

L'objectif du projet est de prédire les points par match d'un joueur lors de
la saison suivante.

Pour une observation correspondant à une saison N :

- les variables explicatives utilisent uniquement les informations disponibles
  jusqu'à la fin de la saison N ;
- la cible `pts_next_season` correspond aux points par match de la saison N+1 ;
- aucune statistique de la saison N+1 n'est utilisée comme variable explicative.

Les features sont créées progressivement selon les catégories proposées dans
le guide : temporelles, agrégées, calculées, catégorielles et indicateurs.

In [31]:
# Variables temporelles de saison

player_dataset[
    "SEASON_START_YEAR"
] = (
    player_dataset[
        "SEASON"
    ]
    .str[:4]
    .astype(int)
)

player_dataset[
    "SEASON_INDEX"
] = (
    player_dataset[
        "SEASON_START_YEAR"
    ]
    - 1996
)

assert player_dataset[
    "SEASON_INDEX"
].min() == 0

assert player_dataset[
    "SEASON_INDEX"
].max() == 26

display(
    player_dataset[
        [
            "SEASON",
            "SEASON_START_YEAR",
            "SEASON_INDEX",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "SEASON_START_YEAR"
    )
)


,SEASON,SEASON_START_YEAR,SEASON_INDEX
0,1996-97,1996,0
441,1997-98,1997,1
880,1998-99,1998,2
1320,1999-00,1999,3
1759,2000-01,2000,4
2200,2001-02,2001,5
2640,2002-03,2002,6
3068,2003-04,2003,7
3510,2004-05,2004,8
3974,2005-06,2005,9


In [32]:
# Création de la cible de la saison suivante

player_dataset = (
    player_dataset
    .sort_values(
        [
            "PLAYER_ID",
            "SEASON_START_YEAR",
        ]
    )
    .reset_index(drop=True)
)


player_dataset[
    "NEXT_SEASON_START_YEAR"
] = (
    player_dataset
    .groupby(
        "PLAYER_ID"
    )[
        "SEASON_START_YEAR"
    ]
    .shift(-1)
)


player_dataset[
    "NEXT_OBSERVED_PTS"
] = (
    player_dataset
    .groupby(
        "PLAYER_ID"
    )[
        "PTS"
    ]
    .shift(-1)
)


consecutive_next_season = (
    player_dataset[
        "NEXT_SEASON_START_YEAR"
    ]
    ==
    player_dataset[
        "SEASON_START_YEAR"
    ]
    + 1
)


player_dataset[
    "pts_next_season"
] = (
    player_dataset[
        "NEXT_OBSERVED_PTS"
    ]
    .where(
        consecutive_next_season
    )
)

In [33]:
# Validation de la cible

target_summary = pd.DataFrame([
    {
        "element":
            "Joueur-saisons totaux",
        "valeur":
            len(player_dataset),
    },
    {
        "element":
            "Cibles disponibles",
        "valeur":
            int(
                player_dataset[
                    "pts_next_season"
                ]
                .notna()
                .sum()
            ),
    },
    {
        "element":
            "Cibles absentes",
        "valeur":
            int(
                player_dataset[
                    "pts_next_season"
                ]
                .isna()
                .sum()
            ),
    },
    {
        "element":
            "Cibles disponibles en 2022-23",
        "valeur":
            int(
                player_dataset.loc[
                    player_dataset[
                        "SEASON"
                    ]
                    == "2022-23",
                    "pts_next_season",
                ]
                .notna()
                .sum()
            ),
    },
])


display(
    target_summary
)

,element,valeur
0,Joueur-saisons totaux,12846
1,Cibles disponibles,9911
2,Cibles absentes,2935
3,Cibles disponibles en 2022-23,0


In [34]:
assert (
    player_dataset.loc[
        player_dataset[
            "SEASON"
        ]
        == "2022-23",
        "pts_next_season",
    ]
    .isna()
    .all()
)


assert (
    player_dataset.loc[
        player_dataset[
            "pts_next_season"
        ]
        .notna(),
        "NEXT_SEASON_START_YEAR",
    ]
    ==
    player_dataset.loc[
        player_dataset[
            "pts_next_season"
        ]
        .notna(),
        "SEASON_START_YEAR",
    ]
    + 1
).all()


print(
    "La cible respecte strictement la saison N+1."
)

La cible respecte strictement la saison N+1.


In [35]:
# Suppression des colonnes intermédiaires

player_dataset = player_dataset.drop(
    columns=[
        "NEXT_OBSERVED_PTS",
        "NEXT_SEASON_START_YEAR",
    ]
)


print(
    "Dimensions après création des premières features :",
    player_dataset.shape
)

Dimensions après création des premières features : (12846, 233)


### Features historiques

Des variables historiques sont créées afin de représenter l'évolution récente
du joueur.

Pour une observation de saison N, seules les informations de la saison N et
des saisons antérieures sont utilisées.

Une saison antérieure n'est considérée comme N-1 que si elle est réellement
consécutive. Une interruption dans la carrière n'est donc pas assimilée à une
saison précédente.

In [36]:
# Saison précédente réellement consécutive

player_dataset = (
    player_dataset
    .sort_values(
        [
            "PLAYER_ID",
            "SEASON_START_YEAR",
        ]
    )
    .reset_index(drop=True)
)


previous_year = (
    player_dataset
    .groupby(
        "PLAYER_ID"
    )[
        "SEASON_START_YEAR"
    ]
    .shift(1)
)


has_previous_season = (
    previous_year
    ==
    player_dataset[
        "SEASON_START_YEAR"
    ]
    - 1
)


player_dataset[
    "HAS_PREVIOUS_SEASON"
] = (
    has_previous_season
    .astype("Int8")
)

In [37]:
# Statistiques de la saison précédente

historical_features = [
    "PTS",
    "MIN",
    "AST",
    "REB",
]


if "USG_PCT" in player_dataset.columns:
    historical_features.append(
        "USG_PCT"
    )


for col in historical_features:

    previous_value = (
        player_dataset
        .groupby(
            "PLAYER_ID"
        )[col]
        .shift(1)
    )

    player_dataset[
        f"{col}_PREV"
    ] = previous_value.where(
        has_previous_season
    )


print(
    "Features historiques créées :",
    [
        f"{col}_PREV"
        for col in historical_features
    ]
)

Features historiques créées : ['PTS_PREV', 'MIN_PREV', 'AST_PREV', 'REB_PREV', 'USG_PCT_PREV']


In [38]:
# Evolution par rapport à la saison précédente

for col in historical_features:

    player_dataset[
        f"{col}_CHANGE"
    ] = (
        player_dataset[col]
        - player_dataset[
            f"{col}_PREV"
        ]
    )


display(
    player_dataset[
        [
            "PLAYER_ID",
            "PLAYER_NAME",
            "SEASON",
            "PTS",
            "PTS_PREV",
            "PTS_CHANGE",
            "HAS_PREVIOUS_SEASON",
        ]
    ]
    .head(20)
)

,PLAYER_ID,PLAYER_NAME,SEASON,PTS,PTS_PREV,PTS_CHANGE,HAS_PREVIOUS_SEASON
0,2,Byron Scott,1996-97,6.7,NaN,NaN,0
1,3,Grant Long,1996-97,5.0,NaN,NaN,0
2,3,Grant Long,1997-98,3.5,5.0,-1.5,1
3,3,Grant Long,1998-99,9.8,3.5,6.3,1
4,3,Grant Long,1999-00,4.8,9.8,-5.0,1
5,3,Grant Long,2000-01,6.0,4.8,1.2,1
6,3,Grant Long,2001-02,6.3,6.0,0.3,1
7,3,Grant Long,2002-03,1.8,6.3,-4.5,1
8,7,Dan Schayes,1996-97,3.0,NaN,NaN,0
9,7,Dan Schayes,1997-98,5.5,3.0,2.5,1


### Features calculées

Des ratios simples sont créés à partir des statistiques de la saison courante.

Ces variables permettent de représenter le volume offensif, l'efficacité et
l'utilisation du joueur indépendamment de certaines statistiques brutes.

Les divisions par zéro sont explicitement contrôlées.

In [39]:
# Fonction de division sécurisée

def safe_divide(numerator, denominator):

    return np.where(
        denominator > 0,
        numerator / denominator,
        np.nan
    )

In [40]:
# Features de volume et d'efficacité

player_dataset[
    "PTS_PER_MIN"
] = safe_divide(
    player_dataset["PTS"],
    player_dataset["MIN"]
)


player_dataset[
    "FGA_PER_MIN"
] = safe_divide(
    player_dataset["FGA"],
    player_dataset["MIN"]
)


player_dataset[
    "FG3A_RATE"
] = safe_divide(
    player_dataset["FG3A"],
    player_dataset["FGA"]
)


player_dataset[
    "FTA_RATE"
] = safe_divide(
    player_dataset["FTA"],
    player_dataset["FGA"]
)


player_dataset[
    "AST_TO_TOV"
] = safe_divide(
    player_dataset["AST"],
    player_dataset["TOV"]
)

In [41]:
calculated_features = [
    "PTS_PER_MIN",
    "FGA_PER_MIN",
    "FG3A_RATE",
    "FTA_RATE",
    "AST_TO_TOV",
]


calculated_features_summary = (
    player_dataset[
        calculated_features
    ]
    .describe()
    .T
)


display(
    calculated_features_summary
)

,count,mean,std,min,25%,50%,75%,max
PTS_PER_MIN,12846.0,0.373792,0.148339,0.0,0.286720,0.364689,0.449664,6.0
FGA_PER_MIN,12846.0,0.321180,0.101940,0.0,0.254985,0.315358,0.380182,2.0
FG3A_RATE,12795.0,0.250584,0.220370,0.0,0.019231,0.239437,0.414149,1.0
FTA_RATE,12795.0,0.294175,0.213101,0.0,0.176471,0.260870,0.367647,5.0
AST_TO_TOV,12512.0,1.497887,0.950434,0.0,0.857143,1.333333,2.000000,20.0


### Features agrégées historiques

Des moyennes historiques sont calculées pour résumer le niveau passé du joueur.

Pour une saison N, les agrégations utilisent uniquement les saisons strictement
antérieures à N.

La saison courante et les saisons futures ne sont pas utilisées dans ces
moyennes historiques.

In [42]:
# Moyennes des saisons antérieures

aggregate_features = [
    "PTS",
    "MIN",
    "AST",
    "REB",
]


for col in aggregate_features:

    player_dataset[
        f"{col}_CAREER_AVG_PREV"
    ] = (
        player_dataset
        .groupby(
            "PLAYER_ID"
        )[col]
        .transform(
            lambda s:
                s.shift(1)
                .expanding()
                .mean()
        )
    )

In [43]:
display(
    player_dataset[
        [
            "PLAYER_ID",
            "PLAYER_NAME",
            "SEASON",
            "PTS",
            "PTS_PREV",
            "PTS_CAREER_AVG_PREV",
        ]
    ]
    .head(25)
)

,PLAYER_ID,PLAYER_NAME,SEASON,PTS,PTS_PREV,PTS_CAREER_AVG_PREV
0,2,Byron Scott,1996-97,6.7,NaN,NaN
1,3,Grant Long,1996-97,5.0,NaN,NaN
2,3,Grant Long,1997-98,3.5,5.0,5.000000
3,3,Grant Long,1998-99,9.8,3.5,4.250000
4,3,Grant Long,1999-00,4.8,9.8,6.100000
5,3,Grant Long,2000-01,6.0,4.8,5.775000
6,3,Grant Long,2001-02,6.3,6.0,5.820000
7,3,Grant Long,2002-03,1.8,6.3,5.900000
8,7,Dan Schayes,1996-97,3.0,NaN,NaN
9,7,Dan Schayes,1997-98,5.5,3.0,3.000000


### Features catégorielles

Des segmentations simples sont créées à partir de l'âge et de l'expérience.

Les bornes sont définies manuellement afin de ne pas apprendre les seuils à
partir de l'ensemble du dataset avant la séparation train/test.

Ces catégories permettent de représenter différentes étapes de la carrière
d'un joueur.

In [44]:
# Catégorie d'âge

player_dataset[
    "AGE_GROUP"
] = pd.cut(
    player_dataset["AGE"],
    bins=[
        -np.inf,
        22,
        27,
        31,
        np.inf,
    ],
    labels=[
        "22_or_less",
        "23_27",
        "28_31",
        "32_plus",
    ],
)


age_group_distribution = (
    player_dataset[
        "AGE_GROUP"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


display(
    age_group_distribution
)

AGE_GROUP
22_or_less    1748
23_27         5822
28_31         3088
32_plus       2188
Name: count, dtype: int64

In [45]:
# Catégorie d'expérience

player_dataset[
    "EXPERIENCE_GROUP"
] = pd.cut(
    player_dataset[
        "EXP_YEARS"
    ],
    bins=[
        -np.inf,
        0,
        3,
        7,
        np.inf,
    ],
    labels=[
        "rookie",
        "1_3_years",
        "4_7_years",
        "8_plus_years",
    ],
)


experience_group_distribution = (
    player_dataset[
        "EXPERIENCE_GROUP"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


display(
    experience_group_distribution
)

EXPERIENCE_GROUP
rookie          1825
1_3_years       3743
4_7_years       3148
8_plus_years    2728
NaN             1402
Name: count, dtype: int64

### Features indicateurs

Des indicateurs binaires sont ajoutés pour représenter certaines situations
sportives importantes.

Ils sont construits uniquement à partir d'informations disponibles pendant ou
avant la saison courante.

In [46]:
# Indicateurs métier

player_dataset[
    "IS_ROOKIE"
] = (
    player_dataset[
        "EXP_YEARS"
    ]
    == 0
).astype("Int8")


player_dataset.loc[
    player_dataset[
        "EXP_YEARS"
    ].isna(),
    "IS_ROOKIE"
] = pd.NA


player_dataset[
    "MULTI_TEAM_SEASON"
] = (
    player_dataset[
        "TEAM_COUNT"
    ] > 1
).astype("Int8")

In [47]:
# Vérification des indicateurs

indicator_summary = pd.DataFrame([
    {
        "feature": "HAS_PREVIOUS_SEASON",
        "valeur_1": int(
            (
                player_dataset[
                    "HAS_PREVIOUS_SEASON"
                ] == 1
            ).sum()
        ),
        "manquants": int(
            player_dataset[
                "HAS_PREVIOUS_SEASON"
            ].isna().sum()
        ),
    },
    {
        "feature": "IS_ROOKIE",
        "valeur_1": int(
            (
                player_dataset[
                    "IS_ROOKIE"
                ] == 1
            ).sum()
        ),
        "manquants": int(
            player_dataset[
                "IS_ROOKIE"
            ].isna().sum()
        ),
    },
    {
        "feature": "MULTI_TEAM_SEASON",
        "valeur_1": int(
            (
                player_dataset[
                    "MULTI_TEAM_SEASON"
                ] == 1
            ).sum()
        ),
        "manquants": int(
            player_dataset[
                "MULTI_TEAM_SEASON"
            ].isna().sum()
        ),
    },
    {
        "feature": "ROSTER_AVAILABLE",
        "valeur_1": int(
            (
                player_dataset[
                    "ROSTER_AVAILABLE"
                ] == 1
            ).sum()
        ),
        "manquants": int(
            player_dataset[
                "ROSTER_AVAILABLE"
            ].isna().sum()
        ),
    },
])


display(
    indicator_summary
)

,feature,valeur_1,manquants
0,HAS_PREVIOUS_SEASON,9911,0
1,IS_ROOKIE,1825,1402
2,MULTI_TEAM_SEASON,1551,0
3,ROSTER_AVAILABLE,11444,0


## 3. Sélection des colonnes

Après les jointures et le Feature Engineering, certaines colonnes sont redondantes ou uniquement techniques.

La sélection finale de cette phase vise à :

- supprimer les colonnes techniques sans valeur analytique ;
- conserver les représentations standardisées utiles ;
- retirer les représentations physiques brutes devenues redondantes ;
- organiser les colonnes de façon logique ;
- conserver un dataset analytique enrichi pour l'EDA et la préparation du ML.

Ce dataset n'est pas encore la matrice `X` définitive du modèle. La sélection des variables réellement utilisées par les modèles sera réalisée après l'EDA et dans la préparation de la Phase 7, sans utiliser le jeu de test pour choisir les features.


In [48]:
# Colonnes techniques ou redondantes

technical_source_columns = [
    col
    for col in player_dataset.columns
    if "SOURCE_FILE" in col
]

columns_to_remove = [
    "PLAYER_HEIGHT",
    "PLAYER_HEIGHT_INCHES",
    "PLAYER_WEIGHT",
    "TEAM_TEAM_NAME",
    "TEAM_TEAM_ABBREVIATION",
]

columns_to_remove = sorted(
    set(
        technical_source_columns
        + [
            col
            for col in columns_to_remove
            if col in player_dataset.columns
        ]
    )
)

print(
    "Colonnes supprimées :",
    len(columns_to_remove)
)

print(
    columns_to_remove
)


Colonnes supprimées : 6
['PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'SOURCE_FILE', 'TEAM_SOURCE_FILE', 'TEAM_TEAM_NAME']


In [49]:
# Suppression des colonnes techniques et redondantes

final_dataset = (
    player_dataset
    .drop(
        columns=columns_to_remove
    )
    .copy()
)


print(
    "Dimensions avant sélection :",
    player_dataset.shape
)

print(
    "Dimensions après sélection :",
    final_dataset.shape
)

Dimensions avant sélection : (12846, 257)
Dimensions après sélection : (12846, 251)


In [50]:
# Organisation des colonnes

priority_columns = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "SEASON",
    "SEASON_START_YEAR",
    "SEASON_INDEX",
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "AGE",
    "AGE_GROUP",
    "POSITION",
    "EXP_YEARS",
    "EXPERIENCE_GROUP",
    "ROSTER_AVAILABLE",
    "HAS_PREVIOUS_SEASON",
    "MULTI_TEAM_SEASON",
    "IS_ROOKIE",
]


target_columns = [
    "pts_next_season",
]


priority_columns = [
    col
    for col in priority_columns
    if col in final_dataset.columns
]


target_columns = [
    col
    for col in target_columns
    if col in final_dataset.columns
]


remaining_columns = [
    col
    for col in final_dataset.columns
    if (
        col not in priority_columns
        and col not in target_columns
    )
]


final_dataset = final_dataset[
    priority_columns
    + remaining_columns
    + target_columns
]


print(
    "Dimensions du dataset final :",
    final_dataset.shape
)


display(
    final_dataset.head()
)

Dimensions du dataset final : (12846, 251)


,PLAYER_ID,PLAYER_NAME,SEASON,SEASON_START_YEAR,SEASON_INDEX,TEAM_ID,TEAM_ABBREVIATION,AGE,AGE_GROUP,POSITION,EXP_YEARS,EXPERIENCE_GROUP,ROSTER_AVAILABLE,HAS_PREVIOUS_SEASON,MULTI_TEAM_SEASON,IS_ROOKIE,NICKNAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,...,TEAM_BLK_RANK,TEAM_BLKA_RANK,TEAM_PF_RANK,TEAM_PFD_RANK,TEAM_PTS_RANK,TEAM_PLUS_MINUS_RANK,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,TEAM_OFF_RATING_RANK,TEAM_DEF_RATING_RANK,TEAM_NET_RATING_RANK,TEAM_AST_PCT_RANK,TEAM_AST_TO_RANK,TEAM_AST_RATIO_RANK,TEAM_OREB_PCT_RANK,TEAM_DREB_PCT_RANK,TEAM_REB_PCT_RANK,TEAM_TM_TOV_PCT_RANK,TEAM_EFG_PCT_RANK,TEAM_TS_PCT_RANK,TEAM_PACE_RANK,TEAM_PIE_RANK,PTS_PREV,MIN_PREV,AST_PREV,REB_PREV,USG_PCT_PREV,PTS_CHANGE,MIN_CHANGE,AST_CHANGE,REB_CHANGE,USG_PCT_CHANGE,PTS_PER_MIN,FGA_PER_MIN,FG3A_RATE,FTA_RATE,AST_TO_TOV,PTS_CAREER_AVG_PREV,MIN_CAREER_AVG_PREV,AST_CAREER_AVG_PREV,REB_CAREER_AVG_PREV,pts_next_season
0,2,Byron Scott,1996-97,1996,0,1610612747,LAL,36.0,32_plus,G,13,8_plus_years,1,0,0,0,Byron,79,54,25,0.684,18.2,2.1,4.8,0.430,0.9,2.4,0.388,1.6,1.9,0.841,0.3,1.2,1.5,1.3,0.7,0.6,0.2,0.1,0.9,0.0,6.7,2.5,12.0,0,0,11.9,86,34,218,95,222,238,242,243,94,103,74,158,179,...,1,11,16,21,8,8,105.2,106.4,100.8,102.3,4.3,4.2,0.611,1.51,17.0,0.343,0.658,0.501,0.159,0.496,0.535,93.8,92.61,77.18,7705,0.530,9,8,9,14,6,13,9,18,14,4,14,16,11,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.368132,0.263736,0.500000,0.395833,1.857143,NaN,NaN,NaN,NaN,NaN
1,3,Grant Long,1996-97,1996,0,1610612765,DET,31.0,28_31,F,8,8_plus_years,1,0,0,0,Grant,65,41,24,0.631,18.0,1.9,4.2,0.447,0.3,0.7,0.362,1.0,1.3,0.750,1.4,2.1,3.4,0.6,0.7,0.7,0.1,0.3,1.6,0.0,5.0,1.1,11.5,2,0,10.8,196,101,233,136,227,257,267,187,205,217,118,244,258,...,29,1,5,9,24,6,107.9,108.5,101.3,102.6,6.6,5.9,0.550,1.49,16.1,0.302,0.675,0.492,0.146,0.512,0.554,86.9,86.09,71.74,7120,0.530,5,9,6,28,8,23,28,8,22,1,4,5,28,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.277778,0.233333,0.166667,0.309524,0.857143,NaN,NaN,NaN,NaN,3.5
2,3,Grant Long,1997-98,1997,1,1610612765,DET,32.0,32_plus,NaN,<NA>,NaN,0,1,0,<NA>,Grant,40,19,21,0.475,18.5,1.3,2.9,0.427,0.0,0.1,0.000,1.0,1.4,0.719,1.4,2.3,3.8,0.6,0.6,0.7,0.3,0.3,2.3,0.1,3.5,-0.1,11.5,0,0,10.0,313,266,255,262,230,326,329,254,274,273,274,249,249,...,25,12,9,17,20,13,102.5,103.4,100.4,101.8,2.1,1.6,0.558,1.33,15.5,0.332,0.668,0.500,0.160,0.472,0.523,90.8,89.75,74.79,7468,0.505,14,9,14,25,19,24,18,12,16,9,20,16,23,16,5.0,18.0,0.6,3.4,0.151,-1.5,0.5,0.0,0.4,-0.048,0.189189,0.156757,0.034483,0.482759,1.000000,5.00,18.000000,0.600000,3.400000,9.8
3,3,Grant Long,1998-99,1998,2,1610612737,ATL,33.0,32_plus,F,10,8_plus_years,1,1,0,0,Grant,50,31,19,0.620,27.6,3.0,7.2,0.421,0.1,0.4,0.167,3.7,4.7,0.783,2.0,3.9,5.9,1.1,1.5,1.1,0.3,0.5,2.9,0.0,9.8,4.1,21.4,5,0,19.7,1,50,180,114,125,168,166,230,221,216,229,34,43,...,13,13,3,5,28,8,96.9,98.4,93.9,95.3,3.0,3.1,0.507,1.05,13.0,0.353,0.701,0.527,0.170,0.436,0.492,88.4,87.10,72.58,4387,0.525,21,2,8,28,28,29,5,1,1,14,26,25,28,8,3.5,18.5,0.6,3.8,0.103,6.3,9.1,0.5,2.1,0.081,0.355072,0.260870,0.055556,0.652778,0.733333,4.25,18.250000,0.600000,3.600000,4.8
4,3,Grant Long,1999-00,1999,3,1610612763,VAN,34.0,32_plus,NaN,<NA>,NaN,0,1,0,<NA>,Grant,42,13,29,0.310,22.0,1.8,4.0,0.443,0.0,0.1,0.000,1.3,1.7,0.775,2.0,3.5,5.6,1.0,1.2,1.1,0.2,0.2,2.6,0.7,4.8,-1.5,15.8,0,0,14.0,310,327,190,352,190,292,301,188,281,277,281,199,212,...,26,28,12,15,26,26,99.7,100.7,105.3,106.7,-5.6,-6.0,0.588,1.23,16.2,0.326,0.679,0.501,0.181,0.474,0.524,93.5,92.46,77.05,7647,0.457,23,28,26,20,25,21,11,20,15,2

### Dataset complet et dataset supervisé

Deux versions sont conservées :

- `final_dataset` contient toutes les observations joueur-saison et peut être
  utilisé pour les analyses descriptives ;
- `supervised_dataset` contient uniquement les observations disposant d'une
  cible `pts_next_season` et sera utilisé pour la modélisation supervisée.

La suppression des lignes sans cible n'est pas une imputation et ne modifie pas
les variables explicatives.

In [51]:
# Dataset supervisé

supervised_dataset = (
    final_dataset[
        final_dataset[
            "pts_next_season"
        ].notna()
    ]
    .copy()
)


print(
    "Dataset complet :",
    final_dataset.shape
)

print(
    "Dataset supervisé :",
    supervised_dataset.shape
)

Dataset complet : (12846, 251)
Dataset supervisé : (9911, 251)


In [52]:
# Vérification anti-leakage et colonnes techniques

forbidden_columns = [
    "NEXT_OBSERVED_PTS",
    "NEXT_SEASON_START_YEAR",
    "HAS_NEXT_SEASON_TARGET",
]

leakage_columns_present = [
    col
    for col in forbidden_columns
    if col in final_dataset.columns
]

technical_columns_present = [
    col
    for col in final_dataset.columns
    if (
        "SOURCE_FILE" in col
        or col.startswith("_merge")
        or col.endswith("_x")
        or col.endswith("_y")
    )
]

print(
    "Colonnes futures présentes :",
    leakage_columns_present
)

print(
    "Colonnes techniques résiduelles :",
    technical_columns_present
)

assert len(
    leakage_columns_present
) == 0

assert len(
    technical_columns_present
) == 0

print(
    "Aucune fuite temporelle ni colonne technique résiduelle détectée."
)


Colonnes futures présentes : []
Colonnes techniques résiduelles : []
Aucune fuite temporelle ni colonne technique résiduelle détectée.


In [53]:
# Validation du dataset final

duplicate_final_keys = int(
    final_dataset.duplicated(
        subset=[
            "PLAYER_ID",
            "SEASON",
        ]
    ).sum()
)

print(
    "Dataset complet :",
    final_dataset.shape
)

print(
    "Dataset supervisé :",
    supervised_dataset.shape
)

print(
    "Doublons PLAYER_ID + SEASON :",
    duplicate_final_keys
)

assert duplicate_final_keys == 0
assert len(final_dataset) == 12846
assert len(supervised_dataset) == 9911
assert final_dataset.shape[1] == 251
assert supervised_dataset.shape[1] == 251
assert final_dataset.columns[-1] == "pts_next_season"


Dataset complet : (12846, 251)
Dataset supervisé : (9911, 251)
Doublons PLAYER_ID + SEASON : 0


## 4. Documentation du dataset final

Le dataset final est documenté afin d'identifier clairement :

- les features dérivées créées en Phase 4 ;
- leur catégorie, leur logique de calcul et leur objectif ;
- l'origine et la transformation des colonnes finales ;
- les valeurs manquantes restantes ;
- la cible utilisée pour la modélisation.

Les 27 features dérivées ci-dessous sont des variables candidates pour les analyses
et la modélisation. Elles ne constituent pas à elles seules la liste définitive des
features du futur modèle : les variables originales pertinentes restent également
candidates, tandis que les identifiants, la cible et les colonnes non adaptées seront
écartés lors de la préparation de la modélisation.


In [54]:
# Inventaire des features dérivées créées

feature_categories = {
    "Temporelles": [
        "SEASON_START_YEAR",
        "SEASON_INDEX",
    ],
    "Historiques": [
        "PTS_PREV",
        "MIN_PREV",
        "AST_PREV",
        "REB_PREV",
        "USG_PCT_PREV",
        "PTS_CHANGE",
        "MIN_CHANGE",
        "AST_CHANGE",
        "REB_CHANGE",
        "USG_PCT_CHANGE",
    ],
    "Calculées": [
        "PTS_PER_MIN",
        "FGA_PER_MIN",
        "FG3A_RATE",
        "FTA_RATE",
        "AST_TO_TOV",
    ],
    "Agrégées": [
        "PTS_CAREER_AVG_PREV",
        "MIN_CAREER_AVG_PREV",
        "AST_CAREER_AVG_PREV",
        "REB_CAREER_AVG_PREV",
    ],
    "Catégorielles": [
        "AGE_GROUP",
        "EXPERIENCE_GROUP",
    ],
    "Indicateurs": [
        "HAS_PREVIOUS_SEASON",
        "IS_ROOKIE",
        "MULTI_TEAM_SEASON",
        "ROSTER_AVAILABLE",
    ],
}

feature_inventory_rows = []

for category, columns in feature_categories.items():
    existing_columns = [
        col
        for col in columns
        if col in final_dataset.columns
    ]

    feature_inventory_rows.append({
        "categorie": category,
        "nombre": len(existing_columns),
        "colonnes": ", ".join(existing_columns),
    })

feature_inventory = pd.DataFrame(
    feature_inventory_rows
)

derived_feature_names = [
    col
    for columns in feature_categories.values()
    for col in columns
]

assert len(derived_feature_names) == 27
assert all(
    col in final_dataset.columns
    for col in derived_feature_names
)

display(
    feature_inventory
)

print(
    "Nombre total de features dérivées :",
    len(derived_feature_names)
)


,categorie,nombre,colonnes
0,Temporelles,2,"SEASON_START_YEAR, SEASON_INDEX"
1,Historiques,10,"PTS_PREV, MIN_PREV, AST_PREV, REB_PREV, USG_PCT_PREV, PTS_CHANGE, MIN_CHANGE, AST_CHANGE, REB_CHANGE, USG_PCT_CHANGE"
2,Calculées,5,"PTS_PER_MIN, FGA_PER_MIN, FG3A_RATE, FTA_RATE, AST_TO_TOV"
3,Agrégées,4,"PTS_CAREER_AVG_PREV, MIN_CAREER_AVG_PREV, AST_CAREER_AVG_PREV, REB_CAREER_AVG_PREV"
4,Catégorielles,2,"AGE_GROUP, EXPERIENCE_GROUP"
5,Indicateurs,4,"HAS_PREVIOUS_SEASON, IS_ROOKIE, MULTI_TEAM_SEASON, ROSTER_AVAILABLE"


Nombre total de features dérivées : 27


### Documentation des features dérivées créées en Phase 4

Chaque feature créée est documentée avec sa logique de calcul et son intérêt pour
la prédiction des points par match de la saison suivante.


In [55]:
# Description des features dérivées

feature_purpose_rows = [
    {
        "feature": "SEASON_START_YEAR",
        "categorie": "Temporelle",
        "logique": "Année de début extraite de SEASON",
        "but": "Représenter la position temporelle de chaque saison.",
    },
    {
        "feature": "SEASON_INDEX",
        "categorie": "Temporelle",
        "logique": "SEASON_START_YEAR - 1996",
        "but": "Représenter l'évolution historique de la NBA sur la période étudiée.",
    },
    {
        "feature": "PTS_PREV",
        "categorie": "Historique",
        "logique": "PTS de la saison N-1 uniquement si elle est consécutive",
        "but": "Mesurer le niveau offensif récent du joueur.",
    },
    {
        "feature": "MIN_PREV",
        "categorie": "Historique",
        "logique": "MIN de la saison N-1 uniquement si elle est consécutive",
        "but": "Capturer le temps de jeu récent du joueur.",
    },
    {
        "feature": "AST_PREV",
        "categorie": "Historique",
        "logique": "AST de la saison N-1 uniquement si elle est consécutive",
        "but": "Capturer la contribution récente à la création offensive.",
    },
    {
        "feature": "REB_PREV",
        "categorie": "Historique",
        "logique": "REB de la saison N-1 uniquement si elle est consécutive",
        "but": "Représenter le niveau de production récent au rebond.",
    },
    {
        "feature": "USG_PCT_PREV",
        "categorie": "Historique",
        "logique": "USG_PCT de la saison N-1 uniquement si elle est consécutive",
        "but": "Mesurer le rôle offensif récent du joueur.",
    },
    {
        "feature": "PTS_CHANGE",
        "categorie": "Historique",
        "logique": "PTS saison N - PTS saison N-1",
        "but": "Identifier une progression ou une régression offensive récente.",
    },
    {
        "feature": "MIN_CHANGE",
        "categorie": "Historique",
        "logique": "MIN saison N - MIN saison N-1",
        "but": "Mesurer l'évolution récente du temps de jeu.",
    },
    {
        "feature": "AST_CHANGE",
        "categorie": "Historique",
        "logique": "AST saison N - AST saison N-1",
        "but": "Mesurer l'évolution du rôle de créateur.",
    },
    {
        "feature": "REB_CHANGE",
        "categorie": "Historique",
        "logique": "REB saison N - REB saison N-1",
        "but": "Mesurer l'évolution récente de la production au rebond.",
    },
    {
        "feature": "USG_PCT_CHANGE",
        "categorie": "Historique",
        "logique": "USG_PCT saison N - USG_PCT saison N-1",
        "but": "Identifier une évolution du rôle offensif dans l'équipe.",
    },
    {
        "feature": "PTS_PER_MIN",
        "categorie": "Calculée",
        "logique": "PTS / MIN si MIN > 0",
        "but": "Mesurer la production offensive rapportée au temps de jeu.",
    },
    {
        "feature": "FGA_PER_MIN",
        "categorie": "Calculée",
        "logique": "FGA / MIN si MIN > 0",
        "but": "Mesurer le volume de tirs rapporté au temps de jeu.",
    },
    {
        "feature": "FG3A_RATE",
        "categorie": "Calculée",
        "logique": "FG3A / FGA si FGA > 0",
        "but": "Mesurer la part des tirs tentés à trois points.",
    },
    {
        "feature": "FTA_RATE",
        "categorie": "Calculée",
        "logique": "FTA / FGA si FGA > 0",
        "but": "Mesurer la fréquence des lancers francs par rapport aux tirs tentés.",
    },
    {
        "feature": "AST_TO_TOV",
        "categorie": "Calculée",
        "logique": "AST / TOV si TOV > 0",
        "but": "Représenter l'efficacité de la création de jeu.",
    },
    {
        "feature": "PTS_CAREER_AVG_PREV",
        "categorie": "Agrégée",
        "logique": "Moyenne des PTS des saisons antérieures disponibles dans le dataset",
        "but": "Représenter le niveau offensif historique observé du joueur.",
    },
    {
        "feature": "MIN_CAREER_AVG_PREV",
        "categorie": "Agrégée",
        "logique": "Moyenne des MIN des saisons antérieures disponibles dans le dataset",
        "but": "Représenter le temps de jeu historique observé du joueur.",
    },
    {
        "feature": "AST_CAREER_AVG_PREV",
        "categorie": "Agrégée",
        "logique": "Moyenne des AST des saisons antérieures disponibles dans le dataset",
        "but": "Représenter le niveau historique observé de création offensive.",
    },
    {
        "feature": "REB_CAREER_AVG_PREV",
        "categorie": "Agrégée",
        "logique": "Moyenne des REB des saisons antérieures disponibles dans le dataset",
        "but": "Représenter le niveau historique observé au rebond.",
    },
    {
        "feature": "AGE_GROUP",
        "categorie": "Catégorielle",
        "logique": "Tranches fixes : <=22, 23-27, 28-31, >=32 ans",
        "but": "Représenter différentes phases de carrière liées à l'âge.",
    },
    {
        "feature": "EXPERIENCE_GROUP",
        "categorie": "Catégorielle",
        "logique": "Tranches fixes : rookie, 1-3, 4-7, 8+ années",
        "but": "Représenter les principales étapes d'expérience NBA.",
    },
    {
        "feature": "HAS_PREVIOUS_SEASON",
        "categorie": "Indicateur",
        "logique": "1 si une saison N-1 consécutive est disponible, sinon 0",
        "but": "Signaler la disponibilité des features historiques récentes.",
    },
    {
        "feature": "IS_ROOKIE",
        "categorie": "Indicateur",
        "logique": "1 si EXP_YEARS = 0, manquant si EXP_YEARS est manquant",
        "but": "Identifier les joueurs lors de leur première saison NBA.",
    },
    {
        "feature": "MULTI_TEAM_SEASON",
        "categorie": "Indicateur",
        "logique": "1 si TEAM_COUNT > 1",
        "but": "Identifier les saisons durant lesquelles le joueur a changé d'équipe.",
    },
    {
        "feature": "ROSTER_AVAILABLE",
        "categorie": "Indicateur",
        "logique": "1 si une correspondance avec le roster existe, sinon 0",
        "but": "Documenter la disponibilité des informations issues du roster.",
    },
]

feature_purpose = pd.DataFrame(
    feature_purpose_rows
)

assert len(feature_purpose) == 27
assert set(feature_purpose["feature"]) == set(derived_feature_names)

display(
    feature_purpose
)


,feature,categorie,logique,but
0,SEASON_START_YEAR,Temporelle,Année de début extraite de SEASON,Représenter la position temporelle de chaque saison.
1,SEASON_INDEX,Temporelle,SEASON_START_YEAR - 1996,Représenter l'évolution historique de la NBA sur la période étudiée.
2,PTS_PREV,Historique,PTS de la saison N-1 uniquement si elle est consécutive,Mesurer le niveau offensif récent du joueur.
3,MIN_PREV,Historique,MIN de la saison N-1 uniquement si elle est consécutive,Capturer le temps de jeu récent du joueur.
4,AST_PREV,Historique,AST de la saison N-1 uniquement si elle est consécutive,Capturer la contribution récente à la création offensive.
5,REB_PREV,Historique,REB de la saison N-1 uniquement si elle est consécutive,Représenter le niveau de production récent au rebond.
6,USG_PCT_PREV,Historique,USG_PCT de la saison N-1 uniquement si elle est consécutive,Mesurer le rôle offensif récent du joueur.
7,PTS_CHANGE,Historique,PTS saison N - PTS saison N-1,Identifier une progression ou une régression offensive récente.
8,MIN_CHANGE,Historique,MIN saison N - MIN saison N-1,Mesurer l'évolution récente du temps de jeu.
9,AST_CHANGE,Historique,AST saison N - AST saison N-1,Mesurer l'évolution du rôle de créateur.


### Documentation des colonnes finales

Le tableau suivant constitue le dictionnaire technique du dataset final. Les
features dérivées disposent d'une description métier détaillée. Les variables
originales sont rattachées à leur source et à la jointure qui les a intégrées.


In [56]:
# Documentation des colonnes finales

feature_description_map = dict(
    zip(
        feature_purpose["feature"],
        feature_purpose["but"],
    )
)

feature_logic_map = dict(
    zip(
        feature_purpose["feature"],
        feature_purpose["logique"],
    )
)

base_descriptions = {
    "PLAYER_ID": "Identifiant NBA unique du joueur.",
    "PLAYER_NAME": "Nom du joueur.",
    "SEASON": "Saison NBA au format YYYY-YY.",
    "TEAM_ID": "Identifiant NBA de l'équipe associée à la ligne joueur-saison.",
    "TEAM_ABBREVIATION": "Abréviation de l'équipe associée à la ligne joueur-saison.",
    "TEAM_COUNT": "Nombre d'équipes représentées par le joueur pendant la saison.",
    "AGE": "Âge du joueur pendant la saison.",
    "POSITION": "Poste du joueur issu du roster lorsque disponible.",
    "EXP_YEARS": "Nombre d'années d'expérience NBA issu du roster lorsque disponible.",
    "PLAYER_HEIGHT_CM": "Taille du joueur standardisée en centimètres.",
    "PLAYER_WEIGHT_KG": "Poids du joueur standardisé en kilogrammes.",
    "pts_next_season": "Points par match du même joueur lors de la saison N+1 consécutive.",
}

column_documentation_rows = []

for col in final_dataset.columns:
    if col == "pts_next_season":
        source = "Cible dérivée"
        transformation = (
            "Décalage temporel des PTS du même joueur vers la saison N+1, "
            "uniquement si la saison suivante est consécutive"
        )
        description = base_descriptions[col]

    elif col in derived_feature_names:
        source = "Feature Engineering"
        transformation = feature_logic_map[col]
        description = feature_description_map[col]

    elif col in [
        "POSITION",
        "EXP_YEARS",
    ]:
        source = "Rosters"
        transformation = "Jointure sur PLAYER_ID + SEASON"
        description = base_descriptions.get(
            col,
            f"Variable {col} issue du roster."
        )

    elif col in bio_columns_to_add:
        source = "Player Bio"
        transformation = "Jointure sur PLAYER_ID + SEASON"
        description = base_descriptions.get(
            col,
            f"Variable {col} issue des données biographiques NBA."
        )

    elif col in advanced_columns_to_add:
        source = "Statistiques joueur avancées"
        transformation = "Jointure sur PLAYER_ID + SEASON"
        description = base_descriptions.get(
            col,
            f"Statistique joueur avancée {col}."
        )

    elif (
        col.startswith("TEAM_")
        and col not in [
            "TEAM_ID",
            "TEAM_ABBREVIATION",
            "TEAM_COUNT",
        ]
    ):
        source = "Statistiques équipe"
        transformation = "Jointure sur TEAM_ID + SEASON"
        description = base_descriptions.get(
            col,
            f"Statistique équipe {col}."
        )

    else:
        source = "Statistiques joueur de base"
        transformation = "Variable issue de la table principale joueur-saison"
        description = base_descriptions.get(
            col,
            f"Statistique ou attribut joueur {col}."
        )

    column_documentation_rows.append({
        "colonne": col,
        "type": str(final_dataset[col].dtype),
        "description": description,
        "source": source,
        "transformation": transformation,
        "valeurs_manquantes": int(
            final_dataset[col]
            .isna()
            .sum()
        ),
    })

column_documentation = pd.DataFrame(
    column_documentation_rows
)

assert len(column_documentation) == final_dataset.shape[1]
assert column_documentation["description"].notna().all()

display(
    column_documentation.head(30)
)


,colonne,type,description,source,transformation,valeurs_manquantes
0,PLAYER_ID,int64,Identifiant NBA unique du joueur.,Statistiques joueur de base,Variable issue de la table principale joueur-saison,0
1,PLAYER_NAME,string,Nom du joueur.,Statistiques joueur de base,Variable issue de la table principale joueur-saison,0
2,SEASON,string,Saison NBA au format YYYY-YY.,Statistiques joueur de base,Variable issue de la table principale joueur-saison,0
3,SEASON_START_YEAR,int64,Représenter la position temporelle de chaque saison.,Feature Engineering,Année de début extraite de SEASON,0
4,SEASON_INDEX,int64,Représenter l'évolution historique de la NBA sur la période étudiée.,Feature Engineering,SEASON_START_YEAR - 1996,0
5,TEAM_ID,int64,Identifiant NBA de l'équipe associée à la ligne joueur-saison.,Statistiques joueur de base,Variable issue de la table principale joueur-saison,0
6,TEAM_ABBREVIATION,object,Abréviation de l'équipe associée à la ligne joueur-saison.,Statistiques joueur de base,Variable issue de la table principale joueur-saison,0
7,AGE,float64,Âge du joueur pendant la saison.,Statistiques joueur de base,Variable issue de la table principale joueur-saison,0
8,AGE_GROUP,category,Représenter différentes phases de carrière liées à l'âge.,Feature Engineering,"Tranches fixes : <=22, 23-27, 28-31, >=32 ans",0
9,POSITION,object,Poste du joueur issu du roster lorsque disponible.,Rosters,Jointure sur PLAYER_ID + SEASON,1402


### Colonnes originales et variables créées

Cette synthèse distingue les variables issues des sources et jointures des
features dérivées créées pendant la Phase 4.


In [57]:
# Colonnes originales et variables créées

original_documentation = column_documentation[
    ~column_documentation["colonne"].isin(
        derived_feature_names
        + ["pts_next_season"]
    )
].copy()

original_rows = []

for source_name, group in original_documentation.groupby(
    "source",
    sort=True,
):
    original_rows.append({
        "categorie": f"Originales - {source_name}",
        "nombre": len(group),
        "exemples": ", ".join(
            group["colonne"]
            .head(5)
            .tolist()
        ),
    })

created_rows = []

for _, row in feature_inventory.iterrows():
    created_rows.append({
        "categorie": f"Features {row['categorie'].lower()}",
        "nombre": int(row["nombre"]),
        "exemples": ", ".join(
            row["colonnes"]
            .split(", ")[:5]
        ),
    })

created_rows.append({
    "categorie": "Cible",
    "nombre": 1,
    "exemples": "pts_next_season",
})

original_vs_created = pd.DataFrame(
    original_rows
    + created_rows
)

assert original_vs_created["nombre"].sum() == final_dataset.shape[1]

display(
    original_vs_created
)


,categorie,nombre,exemples
0,Originales - Player Bio,11,"COLLEGE, COUNTRY, DRAFT_YEAR, DRAFT_ROUND, DRAFT_NUMBER"
1,Originales - Rosters,2,"POSITION, EXP_YEARS"
2,Originales - Statistiques joueur avancées,56,"E_OFF_RATING, OFF_RATING, sp_work_OFF_RATING, E_DEF_RATING, DEF_RATING"
3,Originales - Statistiques joueur de base,68,"PLAYER_ID, PLAYER_NAME, SEASON, TEAM_ID, TEAM_ABBREVIATION"
4,Originales - Statistiques équipe,86,"TEAM_GP, TEAM_W, TEAM_L, TEAM_W_PCT, TEAM_MIN"
5,Features temporelles,2,"SEASON_START_YEAR, SEASON_INDEX"
6,Features historiques,10,"PTS_PREV, MIN_PREV, AST_PREV, REB_PREV, USG_PCT_PREV"
7,Features calculées,5,"PTS_PER_MIN, FGA_PER_MIN, FG3A_RATE, FTA_RATE, AST_TO_TOV"
8,Features agrégées,4,"PTS_CAREER_AVG_PREV, MIN_CAREER_AVG_PREV, AST_CAREER_AVG_PREV, REB_CAREER_AVG_PREV"
9,Features catégorielles,2,"AGE_GROUP, EXPERIENCE_GROUP"


In [58]:
# Description générale du dataset

dataset_description = pd.DataFrame([
    {
        "element": "Nom",
        "valeur": "NBA player-season enriched dataset",
    },
    {
        "element": "Date de création",
        "valeur": pd.Timestamp.now().strftime("%Y-%m-%d"),
    },
    {
        "element": "Granularité",
        "valeur": "1 ligne = 1 PLAYER_ID + SEASON",
    },
    {
        "element": "Nombre de lignes",
        "valeur": len(final_dataset),
    },
    {
        "element": "Nombre de colonnes",
        "valeur": final_dataset.shape[1],
    },
    {
        "element": "Période couverte",
        "valeur": (
            f"{final_dataset['SEASON'].min()} à "
            f"{final_dataset['SEASON'].max()}"
        ),
    },
    {
        "element": "Features dérivées créées",
        "valeur": len(derived_feature_names),
    },
    {
        "element": "Lignes avec cible",
        "valeur": len(supervised_dataset),
    },
    {
        "element": "Cible ML",
        "valeur": "pts_next_season",
    },
])

display(
    dataset_description
)


,element,valeur
0,Nom,NBA player-season enriched dataset
1,Date de création,2026-09-08
2,Granularité,1 ligne = 1 PLAYER_ID + SEASON
3,Nombre de lignes,12846
4,Nombre de colonnes,251
5,Période couverte,1996-97 à 2022-23
6,Features dérivées créées,27
7,Lignes avec cible,9911
8,Cible ML,pts_next_season


In [59]:
# Sauvegarde des datasets et rapports de Phase 4

final_dataset_path = (
    PROCESSED_DIR
    / "nba_player_season_enriched.parquet"
)

supervised_dataset_path = (
    PROCESSED_DIR
    / "nba_player_season_supervised.parquet"
)

final_dataset.to_parquet(
    final_dataset_path,
    index=False
)

supervised_dataset.to_parquet(
    supervised_dataset_path,
    index=False
)

feature_inventory.to_csv(
    REPORT_DIR
    / "03_inventaire_features.csv",
    index=False,
    encoding="utf-8"
)

feature_purpose.to_csv(
    REPORT_DIR
    / "03_description_features_creees.csv",
    index=False,
    encoding="utf-8"
)

column_documentation.to_csv(
    REPORT_DIR
    / "03_documentation_dataset_final.csv",
    index=False,
    encoding="utf-8"
)

original_vs_created.to_csv(
    REPORT_DIR
    / "03_originales_vs_creees.csv",
    index=False,
    encoding="utf-8"
)

dataset_description.to_csv(
    REPORT_DIR
    / "03_description_dataset_final.csv",
    index=False,
    encoding="utf-8"
)

print(
    "Dataset complet :",
    final_dataset_path.resolve()
)

print(
    "Dataset supervisé :",
    supervised_dataset_path.resolve()
)


Dataset complet : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\processed\nba_player_season_enriched.parquet
Dataset supervisé : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\processed\nba_player_season_supervised.parquet


In [60]:
# Validation finale de la Phase 4

phase4_validation = pd.DataFrame([
    {
        "controle": "Granularité PLAYER_ID + SEASON unique",
        "valide": duplicate_final_keys == 0,
    },
    {
        "controle": "12 846 joueur-saisons conservés",
        "valide": len(final_dataset) == 12846,
    },
    {
        "controle": "Dataset final à 251 colonnes",
        "valide": final_dataset.shape[1] == 251,
    },
    {
        "controle": "9 911 observations supervisées",
        "valide": len(supervised_dataset) == 9911,
    },
    {
        "controle": "Cible présente et placée en dernière colonne",
        "valide": (
            "pts_next_season" in final_dataset.columns
            and final_dataset.columns[-1] == "pts_next_season"
        ),
    },
    {
        "controle": "27 features dérivées présentes",
        "valide": (
            len(derived_feature_names) == 27
            and all(
                col in final_dataset.columns
                for col in derived_feature_names
            )
        ),
    },
    {
        "controle": "Aucune colonne future technique",
        "valide": len(leakage_columns_present) == 0,
    },
    {
        "controle": "Aucune colonne technique résiduelle",
        "valide": len(technical_columns_present) == 0,
    },
    {
        "controle": "5 jointures documentées",
        "valide": len(join_report) == 5,
    },
    {
        "controle": "Documentation complète des colonnes",
        "valide": (
            len(column_documentation) == final_dataset.shape[1]
            and column_documentation["description"].notna().all()
        ),
    },
    {
        "controle": "Fichier complet sauvegardé",
        "valide": final_dataset_path.exists(),
    },
    {
        "controle": "Fichier supervisé sauvegardé",
        "valide": supervised_dataset_path.exists(),
    },
])

display(
    phase4_validation
)

assert phase4_validation[
    "valide"
].all()

print(
    "Tous les contrôles de la Phase 4 sont satisfaits."
)


,controle,valide
0,Granularité PLAYER_ID + SEASON unique,True
1,12 846 joueur-saisons conservés,True
2,Dataset final à 251 colonnes,True
3,9 911 observations supervisées,True
4,Cible présente et placée en dernière colonne,True
5,27 features dérivées présentes,True
6,Aucune colonne future technique,True
7,Aucune colonne technique résiduelle,True
8,5 jointures documentées,True
9,Documentation complète des colonnes,True


Tous les contrôles de la Phase 4 sont satisfaits.


### Limites de la transformation

Plusieurs limites sont conservées explicitement dans l'interprétation du dataset :

- les moyennes `*_CAREER_AVG_PREV` utilisent uniquement les saisons antérieures
  disponibles dans le dataset, qui commence en 1996-97. Elles ne représentent donc
  pas toujours l'intégralité de la carrière réelle d'un vétéran déjà actif avant 1996 ;
- le roster n'est pas disponible pour toutes les joueur-saisons. Les lignes concernées
  sont conservées et identifiées grâce à `ROSTER_AVAILABLE` ;
- pour les saisons où un joueur a représenté plusieurs équipes, `MULTI_TEAM_SEASON`
  signale le cas. Les statistiques équipe jointes correspondent à l'équipe associée
  par la source joueur et doivent être interprétées avec cette limite ;
- le dataset Kaggle a été volontairement exclu de la jointure finale car il ne contient
  pas `PLAYER_ID`, présente quelques homonymes sur `player_name + season` et apporte
  principalement des informations déjà disponibles dans les sources NBA API.


## Questions de réflexion

### 1. Feature la plus utile

`PTS_PREV` semble être l'une des features les plus prometteuses car la
production offensive récente d'un joueur constitue une information directement
liée à la production attendue lors de la saison suivante. Cette hypothèse devra
être vérifiée pendant l'EDA analytique et la modélisation.

### 2. Jointures

Aucune ligne n'a été perdue lors des jointures principales grâce à
l'utilisation de jointures `left`.

Le roster ne couvre cependant que 11 444 des 12 846 joueur-saisons. Les
observations sans correspondance ont été conservées et leur disponibilité est
documentée avec `ROSTER_AVAILABLE`.

Le dataset Kaggle n'a pas été intégré en raison de l'absence de `PLAYER_ID`,
de plusieurs homonymes et de la redondance de ses principales statistiques
avec les sources NBA API.

### 3. Équilibre du dataset

Le dataset enrichi contient un grand nombre de variables issues des différentes
sources. Les redondances techniques évidentes ont été retirées, mais une
sélection complémentaire des variables destinées au modèle sera réalisée lors
de la préparation de la modélisation.

### 4. Features manquantes

Certaines informations pourraient enrichir davantage la prédiction, par
exemple les blessures, les changements de rôle prévus, les contrats ou les
mouvements d'effectif avant la saison suivante.

Ces informations ne sont pas disponibles dans les sources actuellement
utilisées.

## Utilisation de l'IA

L'IA a été utilisée comme assistance méthodologique pour :

- proposer et comparer des stratégies de jointure ;
- identifier les risques de multiplication de lignes ;
- suggérer des features adaptées au contexte NBA ;
- vérifier les risques de data leakage ;
- structurer la documentation des transformations.

Les propositions ont ensuite été vérifiées à partir des données et validées
par des contrôles programmatiques avant d'être retenues.

## Conclusion de la Phase 4

Les différentes sources NBA nettoyées ont été combinées afin de construire un
dataset unique au grain `PLAYER_ID + SEASON`.

Cinq jointures sont documentées. Elles combinent les statistiques joueur de base,
les informations biographiques, les statistiques joueur avancées, les statistiques
équipe de base et avancées ainsi que les informations du roster. Les jointures
principales conservent les 12 846 joueur-saisons sans multiplication inattendue de
lignes.

Le dataset Kaggle a été évalué mais volontairement exclu de la jointure finale en
raison de l'absence de `PLAYER_ID`, de plusieurs homonymes et de la redondance de ses
principales statistiques avec les données NBA API.

Au total, 27 features dérivées ont été créées dans six catégories : temporelles,
historiques, calculées, agrégées, catégorielles et indicateurs. Elles complètent les
variables originales des différentes sources et ne constituent pas à elles seules la
future matrice de features du modèle.

La cible `pts_next_season` représente les points par match du même joueur lors de la
saison immédiatement suivante. Sa construction et les features historiques respectent
strictement l'ordre temporel. Aucune imputation statistique, normalisation ou autre
transformation apprise sur l'ensemble des données n'est effectuée pendant cette phase.

Après suppression des colonnes techniques et redondantes identifiées, le dataset
complet contient 12 846 lignes et 251 colonnes. Le dataset supervisé contient 9 911
observations disposant d'une cible.

Le dataset final est sélectionné, ordonné, documenté et sauvegardé. Il est prêt pour
la Phase 5 : EDA Analytique.
